In [106]:
import os
from IPython.display import display, Image, clear_output
import ipywidgets as widgets
from PIL import Image as PILImage
from ipywidgets import Dropdown, Text, Button
from image_creator import ImageVisualizer
import matplotlib.pyplot as plt
import numpy as np
import re
import rasterio
import sys 
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import utils
import matplotlib.colors as colors
from matplotlib.ticker import FuncFormatter

In [107]:
class ImageVisualizer:
    def __init__(self):
        self.band_mappings = {
            "Landsat_5_7": {"RED": 3, "GREEN": 2, "BLUE": 1, "NIR": 4, "TIR": 9},
            "Landsat_8_9": {"RED": 4, "GREEN": 3, "BLUE": 2, "NIR": 5, "TIR": 9}
        }

    def rescale_to_8bit(self, band_array, min_value=None, max_value=None):
        """Rescale array values to 8-bit (0-255)."""
        valid_pixels = band_array[~np.isnan(band_array)]
        if len(valid_pixels) == 0:
            min_value, max_value = 0, 255
        else:
            if min_value is None:
                min_value = np.percentile(valid_pixels, 2)
            if max_value is None:
                max_value = np.percentile(valid_pixels, 98)

        band_array = np.clip(band_array, min_value, max_value)
        band_array = ((band_array - min_value) / (max_value - min_value) * 255).astype(np.uint8)
        return np.nan_to_num(band_array)
    

    def extract_landsat_version(self, filename):
        """Extract Landsat version from filename."""
        match = re.search(r"Landsat\d+", filename)
        if match:
            return match.group(0)
        else:
            raise ValueError(f"Landsat version could not be determined from filename: {filename}")
        
    def extract_pivot_id(self, filename):
        """Extract pivot ID from the filename."""
        match = re.search(r"CP_(\d+)", filename)
        if match:
            return int(match.group(1))  # Ensure pivot ID is returned as an integer
        else:
            raise ValueError(f"Pivot ID could not be determined from filename: {filename}")

    def get_band_mapping(self, landsat_version):
        """Get band mapping for the Landsat version."""
        if landsat_version in ["Landsat5", "Landsat7"]:
            return self.band_mappings["Landsat_5_7"]
        elif landsat_version in ["Landsat8", "Landsat9"]:
            return self.band_mappings["Landsat_8_9"]
        else:
            raise ValueError(f"Unsupported Landsat version: {landsat_version}")

    def generate_rgb_image(self, bands, band_mapping):
        """Generate RGB image from band data."""
        red = self.rescale_to_8bit(bands[band_mapping["RED"] - 1])
        green = self.rescale_to_8bit(bands[band_mapping["GREEN"] - 1])
        blue = self.rescale_to_8bit(bands[band_mapping["BLUE"] - 1])
        return np.stack((red, green, blue), axis=-1)

    def generate_combined_image(self, bands, band_mapping, min_kelvin=280, max_kelvin=320):
        # RGB
        rgb_image = self.generate_rgb_image(bands, band_mapping)

        # NDVI
        nir = bands[band_mapping["NIR"] - 1]
        red = bands[band_mapping["RED"] - 1]
        ndvi = (nir - red) / (nir + red)
        ndvi_image = self.rescale_to_8bit(ndvi, -1, 1)

        # LST: Clip to minimum and rescale explicitly
        lst = bands[band_mapping["TIR"] - 1]
        lst = np.maximum(lst, min_kelvin)
        lst_image = self.rescale_to_8bit(lst, min_kelvin, max_kelvin)

        return rgb_image, ndvi_image, lst_image




In [108]:
# Set directory paths
data_root = utils.get_data_root() 
input_dir = input_dir = os.path.join(data_root, 'intermediate/training_data_C02')
tif_files = [
    os.path.join(input_dir, file)
    for file in sorted(os.listdir(input_dir))
    if file.lower().endswith(".tif")
]

# Initialize ImageVisualizer
visualizer = ImageVisualizer()

# Helper function to load bands from a TIF file
def load_bands(tif_file):
    """Load bands from a TIF file."""
    with rasterio.open(tif_file) as src:
        bands = [src.read(i + 1, masked=True).filled(np.nan) for i in range(src.count)]
    return bands

# Load TIF files and extract metadata (Pivot ID and Landsat version)
tif_metadata = [
    {
        "filename": file,
        "landsat_version": visualizer.extract_landsat_version(os.path.basename(file)),
        "pivot_id": visualizer.extract_pivot_id(os.path.basename(file))  # Add pivot_id here
    }
    for file in tif_files
]


In [ ]:
# Dropdown for filtering by Landsat
landsat_options = list(sorted(set(item["landsat_version"] for item in tif_metadata)))
landsat_filter = Dropdown(options=["All"] + landsat_options, description="Landsat:")

# Text box for searching by Pivot ID
pivot_search = Text(description="Pivot ID:", placeholder="Enter Pivot ID")
search_button = Button(description="Search")

# Output area for visualization
output_area = widgets.Output()

# Buttons for visualization
rgb_button = widgets.Button(description="Show RGB")
combined_button = widgets.Button(description="Show Combined")
next_button = widgets.Button(description="Next Image")
previous_button = widgets.Button(description="Previous Image")

# Image index for navigation
current_index = widgets.IntText(value=0, description="Image Index:", disabled=True)

# Helper functions
def filter_files(selected_landsat, pivot_query):
    """Filter TIF files based on Landsat and Pivot ID."""
    filtered_files = [
        item for item in tif_metadata
        if (selected_landsat == "All" or item["landsat_version"] == selected_landsat)
        and (pivot_query == "" or str(item["pivot_id"]) == pivot_query)
    ]
    return filtered_files

def search_and_filter(b):
    """Filter and display the first matching TIF file."""
    selected_landsat = landsat_filter.value
    pivot_query = pivot_search.value.strip()

    # Get filtered files
    filtered_files = filter_files(selected_landsat, pivot_query)

    if filtered_files:
        # Update the current index to the first matching file
        current_index.value = tif_files.index(filtered_files[0]["filename"])
        show_combined(None)  # Display the first matching file
    else:
        with output_area:
            clear_output(wait=True)
            print("No matching TIF files found.")

# Button handlers for RGB, Combined, and Next Image
def show_rgb(b):
    """Show the RGB visualization."""
    index = current_index.value
    if 0 <= index < len(tif_files):
        tif_file = tif_files[index]
        bands = load_bands(tif_file)
        landsat_version = visualizer.extract_landsat_version(os.path.basename(tif_file))
        band_mapping = visualizer.get_band_mapping(landsat_version)

        rgb_image = visualizer.generate_rgb_image(bands, band_mapping)
        with output_area:
            clear_output(wait=True)
            plt.imshow(rgb_image)
            plt.axis("off")
            plt.title(f"RGB Visualization\n{os.path.basename(tif_file)}")
            plt.show()

def show_combined(b):
    """Show the Combined visualization (RGB, NDVI, LST) with a dynamic Kelvin range colorbar."""
    index = current_index.value
    if 0 <= index < len(tif_files):
        tif_file = tif_files[index]
        bands = load_bands(tif_file)
        landsat_version = visualizer.extract_landsat_version(os.path.basename(tif_file))
        band_mapping = visualizer.get_band_mapping(landsat_version)

        # Generate the RGB visualization
        rgb_image = visualizer.generate_rgb_image(bands, band_mapping)

        # Generate the NDVI visualization
        nir = bands[band_mapping["NIR"] - 1]
        red = bands[band_mapping["RED"] - 1]
        ndvi = (nir - red) / (nir + red)
        ndvi_image = visualizer.rescale_to_8bit(ndvi, -1, 1)

        # Get raw LST (in Kelvin) from the TIR band
        raw_lst = bands[band_mapping["TIR"] - 1]
        # Compute maximum LST value and set dynamic lower bound (max - 40)
        max_val = np.nanmax(raw_lst)
        lower_bound = max_val - 30


        # We'll use a normalization from 0 to 255 for the image display.
        lst_image = visualizer.rescale_to_8bit(raw_lst)
        norm = colors.Normalize(vmin=0, vmax=255)

        with output_area:
            clear_output(wait=True)
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            fig.suptitle(f"Combined Visualization\n{os.path.basename(tif_file)}", fontsize=16)

            # RGB visualization
            axes[0].imshow(rgb_image)
            axes[0].set_title("RGB")
            axes[0].axis("off")

            # NDVI visualization
            axes[1].imshow(ndvi_image, cmap="gray")
            axes[1].set_title("NDVI")
            axes[1].axis("off")

            # LST visualization with dynamic Kelvin range on the colorbar
            im = axes[2].imshow(lst_image, cmap="coolwarm", norm=norm, interpolation="nearest")
            axes[2].set_title("LST (Kelvin)")
            axes[2].axis("off")

            # Create a formatter to map 8-bit values back to Kelvin using the dynamic range.
            def kelvin_formatter(x, pos):
                kelvin = lower_bound + (x / 255) * 30
                return f"{kelvin:.0f}"

            cbar = fig.colorbar(im, ax=axes[2], orientation="vertical", fraction=0.046, pad=0.04)
            cbar.set_label("LST (Kelvin)")
            cbar.ax.yaxis.set_major_formatter(FuncFormatter(kelvin_formatter))

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
            
def show_next_image(b):
    """Advance to the next image."""
    current_index.value += 1
    if current_index.value < len(tif_files):
        clear_output(wait=True)
        print(f"Processing Image {current_index.value + 1} of {len(tif_files)}...")
    else:
        current_index.value -= 1
        clear_output(wait=True)
        print("No more images to display!")

def show_previous_image(b):
    """Advance to the next image."""
    current_index.value -= 1
    if current_index.value < len(tif_files):
        clear_output(wait=True)
        print(f"Processing Image {current_index.value - 1} of {len(tif_files)}...")
    else:
        current_index.value += 1
        clear_output(wait=True)
        print("No more images to display!")

# Assign button handlers
rgb_button.on_click(show_rgb)
combined_button.on_click(show_combined)
next_button.on_click(show_next_image)
previous_button.on_click(show_previous_image)
search_button.on_click(search_and_filter)


Processing Image 6 of 742...


In [110]:
# Layout for the widgets
ui = widgets.VBox([
    widgets.HBox([landsat_filter, pivot_search, search_button]),  # Added new widgets
    widgets.HBox([rgb_button, combined_button, next_button, previous_button]),
    widgets.HBox([current_index]),
    output_area
])

# Display the interactive UI
display(ui)
